In [11]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
import requests
import time
from bs4 import BeautifulSoup
import json
from datetime import datetime
import random
%matplotlib inline

# Import settings from config
from config import (
    BASE_DIR, DATA_DIR, OUTPUT_DIR, 
    TRANSCRIPTS_DIR, SEC_FILINGS_DIR,
    USER_AGENT, SEC_DELAY, SEC_MAX_RETRIES,
    COMPANIES
)

# Helper function for better console output
def status_print(message):
    """Print a status message with timestamp"""
    timestamp = datetime.now().strftime('%H:%M:%S')
    print(f"[{timestamp}] {message}")

# Improved function to fetch SEC data with retry logic
def fetch_with_retry(url, max_retries=SEC_MAX_RETRIES):
    """Fetch URL with exponential backoff retry logic"""
    headers = {
        'User-Agent': USER_AGENT,  # Using your real email
        'Accept-Encoding': 'gzip, deflate',
        'Host': 'www.sec.gov'
    }

    for attempt in range(max_retries):
        try:
            status_print(f"Fetching {url}, attempt {attempt+1}/{max_retries}...")
            response = requests.get(url, headers=headers)

            if response.status_code == 200:
                status_print("Request successful")
                return response
            elif response.status_code == 503:
                wait_time = (2 ** attempt) * SEC_DELAY  # Exponential backoff
                status_print(f"Got 503 error, waiting {wait_time}s before retry...")
                time.sleep(wait_time)
            else:
                status_print(f"Error: HTTP {response.status_code}")
                return None
        except Exception as e:
            status_print(f"Exception: {str(e)}")
            if attempt < max_retries - 1:  # Don't sleep on the last attempt
                time.sleep(SEC_DELAY)

    status_print("All retry attempts failed")
    return None

def fetch_and_parse_8k(ticker, cik, num_filings=5):
    """Fetch and parse 8-K filings for a company"""
    status_print(f"Fetching 8-K filings for {ticker} (CIK: {cik})...")

    # Create the directory for this company's filings if it doesn't exist
    company_dir = os.path.join(SEC_FILINGS_DIR, ticker)
    os.makedirs(company_dir, exist_ok=True)

    # Construct the URL to get the company's filings index
    sec_ticker_url = f"https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK={cik}&type=8-K&count={num_filings}"

    # Request the company's filing index page
    response = fetch_with_retry(sec_ticker_url)
    if not response:
        status_print(f"Failed to fetch 8-K index for {ticker}")
        return []

    # Parse the HTML to find the 8-K filings
    soup = BeautifulSoup(response.text, 'html.parser')
    filing_table = soup.find('table', class_='tableFile2')

    if not filing_table:
        status_print(f"No filings found for {ticker}")
        return []

    # Find all rows in the table
    rows = filing_table.find_all('tr')

    # List to store filing information
    filings = []

    # Skip the header row
    for row in rows[1:]:
        cols = row.find_all('td')
        if len(cols) >= 4:  # Make sure we have enough columns
            filing_type = cols[0].text.strip()

            if '8-K' in filing_type:
                filing_date = cols[3].text.strip()
                filing_desc = cols[2].text.strip() if len(cols) > 2 else ""
                documents_link = cols[1].find('a')['href'] if cols[1].find('a') else None

                if documents_link:
                    # Construct the full URL
                    documents_url = f"https://www.sec.gov{documents_link}"

                    # Get the documents index page
                    time.sleep(SEC_DELAY)  # Important to avoid rate limits
                    doc_response = fetch_with_retry(documents_url)

                    if doc_response:
                        # Find the actual 8-K document link and get its content
                        # (Your existing code to extract and save 8-K content)
                        filing_info = {
                            'ticker': ticker,
                            'filing_date': filing_date,
                            'filing_type': filing_type,
                            'filing_desc': filing_desc,
                            'url': documents_url
                        }
                        filings.append(filing_info)
                        status_print(f"Added 8-K filing from {filing_date} for {ticker}")

    status_print(f"Fetched {len(filings)} 8-K filings for {ticker}")
    return filings

# Function to get transcript files and extract tickers
def get_transcript_tickers(transcripts_dir):
    """Extract unique tickers from transcript filenames"""
    transcript_files = [f for f in os.listdir(transcripts_dir) if f.endswith('.txt')]
    tickers = set()

    ticker_pattern = r'^([A-Z]+)_'

    for filename in transcript_files:
        match = re.match(ticker_pattern, filename)
        if match:
            tickers.add(match.group(1))

    return list(tickers)

# Main function to visualize SEC data with transcript data
def visualize_sec_with_transcripts():
    # Get tickers from transcript files
    transcript_tickers = get_transcript_tickers(TRANSCRIPTS_DIR)
    status_print(f"Found {len(transcript_tickers)} tickers in transcript files: {', '.join(transcript_tickers)}")

    all_filings = []

    # Filter companies list to only include those we have transcripts for
    companies_to_process = [c for c in COMPANIES if c['ticker'] in transcript_tickers]

    for company in companies_to_process:
        ticker = company['ticker']
        cik = company['cik']

        # Get 8-K filings
        filings = fetch_and_parse_8k(ticker, cik, num_filings=5)
        all_filings.extend(filings)

        # Wait between companies to avoid rate limits
        time.sleep(SEC_DELAY)

    if all_filings:
        # Convert to DataFrame
        filings_df = pd.DataFrame(all_filings)

        # Save to CSV
        csv_path = os.path.join(SEC_FILINGS_DIR, "all_8K_filings.csv")
        filings_df.to_csv(csv_path, index=False)
        status_print(f"Saved {len(all_filings)} 8-K filings to {csv_path}")

        # Continue with visualization code...
        # (Use your existing visualization code here)

        return filings_df
    else:
        status_print("No filings found for visualization")
        return None

# Run the visualization function
if __name__ == "__main__":
    status_print("Starting SEC 8-K visualization")
    filings_df = visualize_sec_with_transcripts()
    status_print("SEC 8-K visualization complete")


[22:17:45] Starting SEC 8-K visualization
[22:17:45] Found 23 tickers in transcript files: HD, V, MSFT, JPM, PG, NFLX, UNH, GOOGL, AMZN, BAC, META, WMT, JNJ, PEP, TSLA, KO, PFE, VZ, AAPL, GS, MRK, NVDA, WFC
[22:17:45] Fetching 8-K filings for AAPL (CIK: 0000320193)...
[22:17:45] Fetching https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=0000320193&type=8-K&count=5, attempt 1/3...
[22:17:47] Request successful
[22:17:50] Fetching https://www.sec.gov/Archives/edgar/data/320193/000114036125005876/0001140361-25-005876-index.htm, attempt 1/3...
[22:17:50] Request successful
[22:17:50] Added 8-K filing from 2025-02-25 for AAPL
[22:17:53] Fetching https://www.sec.gov/Archives/edgar/data/320193/000032019325000007/0000320193-25-000007-index.htm, attempt 1/3...
[22:17:53] Request successful
[22:17:53] Added 8-K filing from 2025-01-30 for AAPL
[22:17:56] Fetching https://www.sec.gov/Archives/edgar/data/320193/000114036125000228/0001140361-25-000228-index.htm, attempt 1/3...
[22:17:56